In [1]:
from datasets import load_dataset
import pandas as pd
import re


In [2]:
ds_raw  = load_dataset("MahsaVafaie/BZKopen", "raw")
ds_norm = load_dataset("MahsaVafaie/BZKopen", "normalized")

DATE_COLS = ["ApplicantBirthDate", "VictimBirthDate", "VictimDeathDate"]

def hf_to_df(ds):
    dfs = []
    for split in ["train", "validation", "test"]:
        df = ds[split].to_pandas().drop(columns=["image"])
        df["split"] = split
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

raw_df  = hf_to_df(ds_raw)
norm_df = hf_to_df(ds_norm)

print(f"Total rows: {len(raw_df)}")
print(f"Splits:     {raw_df['split'].value_counts().to_dict()}")
raw_df[["split"] + DATE_COLS].head()


Total rows: 516
Splits:     {'train': 361, 'test': 78, 'validation': 77}


,split,ApplicantBirthDate,VictimBirthDate,VictimDeathDate
0,train,17.4.1819,,
1,train,6.6.1819,,
2,train,,25.8.1829,
3,train,,13.3.50,
4,train,,8.7.50,


In [3]:
pairs = pd.concat([
    raw_df[DATE_COLS].rename(columns=lambda c: c + "_raw"),
    norm_df[DATE_COLS].rename(columns=lambda c: c + "_norm"),
], axis=1)
mask = raw_df[DATE_COLS].apply(lambda col: col.str.len() > 0).any(axis=1)

In [ ]:
GERMAN_MONTHS = {
    'januar': 1,  'jan': 1,   'january': 1,  'j': 1,
    'jänner': 1,  'janner': 1,
    'februar': 2, 'feb': 2,   'febr': 2,     'february': 2,
    'maerz': 3,   'märz': 3,  'mar': 3,      'mär': 3,    'march': 3,
    'april': 4,   'apr': 4,
    'mai': 5,     'may': 5,
    'juni': 6,    'jun': 6,   'june': 6,     'jani': 6,
    'juli': 7,    'jul': 7,   'july': 7,
    'august': 8,  'aug': 8,   'ug': 8,       'augl': 8,
    'september': 9, 'sep': 9, 'sept': 9,
    'oktober': 10, 'okt': 10, 'oct': 10,     'october': 10, 'k': 10, 'ct':10,
    'november': 11, 'nov': 11, 'novl': 11,
    'dezember': 12, 'dez': 12, 'dec': 12,    'december': 12, 'dezemb': 12,
    'elul': 6,   
}

ROMAN = {
    'i': 1, 'ii': 2, 'iii': 3, 'iv': 4, 'v': 5, 'vi': 6,
    'vii': 7, 'viii': 8, 'ix': 9, 'x': 10, 'xi': 11, 'xii': 12,
    'xiii': 13,  
}

NON_DATE_TOKENS = {
    'unbekannt', 'deportiert', 'verstorben', 'verst', 'deportation',
    'umgekommen', 'gestorben', 'beide', 'unk',
}

SEASON_TO_MONTH = {'sommer': 8, 'frühling': 4, 'herbst': 10, 'winter': 1}

_EMPTY_RAW = {'nan', 'none', 'null', '', '-', '?'}

_DATE_TOKEN = re.compile(r'\b(\d{1,2})[.\-/]+\s*(\d{1,2})[.\-/]+\s*(\d{2,4})\b')

# Two 4-digit years joined by connector — take the latest
_ODER_PAT = re.compile(
    r'\b(\d{4})\s*(?:oder|or|bis|und|,)\s*(\d{4})\b', re.IGNORECASE
)

# OCR digit-for-letter fixes applied before all pattern matching
_OCR_FIXES = [
    (re.compile(r'\b0([Kk][Tt])\b'), r'O\1'),   # 0kt → Okt
    (re.compile(r'\b1([Vv])\b'),     r'I\1'),   # 1V  → IV
    (re.compile(r'\b8([Uu][Ll])\b'), r'J\1'),   # 8ul → Jul
]

_UML = r'A-Za-z\u00c4\u00e4\u00d6\u00f6\u00dc\u00fc\u00df'

MANUAL = 'MANUAL'


class _UncertainYear(int):
    """Int subclass marking a guessed century for an ambiguous 2-digit year."""
    def __add__(self, other): return _UncertainYear(int.__add__(self, other))
    def __radd__(self, other): return _UncertainYear(int.__radd__(self, other))


def _maybe_flag(iso, resolved):
    """Prefix iso with '~' when the year was a best-guess (ambiguous 2-digit input)."""
    if iso and isinstance(resolved, _UncertainYear):
        return '~' + iso
    return iso


def _ocr_fix(s):
    for pat, repl in _OCR_FIXES:
        s = pat.sub(repl, s)
    return s


def _parse_roman(s):
    """Parse any Roman numeral string (case-insensitive). Returns 0 on failure."""
    vals = {'i': 1, 'v': 5, 'x': 10, 'l': 50, 'c': 100}
    s = s.lower()
    if not s or not all(c in vals for c in s):
        return 0
    result, prev = 0, 0
    for ch in reversed(s):
        v = vals[ch]
        result += v if v >= prev else -v
        prev = v
    return result


def _century(yy, col):
    if 'DeathDate' in (col or ''):
        return 1900
    if col == 'ApplicantBirthDate':
        if yy >= 55: return 1800
        if yy <= 33: return 1900
        return _UncertainYear(1900)   # 34–54: best-guess, flagged for review (persecution: maily 1933-1945, someone born 1834-1854 will be 79-111 during that time, difficult to be an applicant)
    if col == 'VictimBirthDate':
        if yy >= 45: return 1800
        if yy <= 33: return 1900
        return _UncertainYear(1900)   # 34–44: best-guess, flagged for review (persecution: maily 1933-1945, someone born 1834-1844 will be 89-111 during that time, rare to be a victim)
    return 1800 if yy >= 50 else 1900


def _fix_component(v):
    if v > 31 and v >= 10:
        rev = int(str(v)[::-1])
        if 1 <= rev <= 31:
            return rev
    return v


def _to_iso(day, month, year, partial=False):
    try:
        d, m, y = _fix_component(int(day)), int(month), int(year)
        m = _fix_component(m)
        if m > 12 and d <= 12:
            d, m = m, d
        if 1 <= m <= 12 and 1 <= d <= 31:
            return f'{y:04d}-{m:02d}-{d:02d}'
        if partial:
            return f'{y:04d}-{m:02d}-01' if 1 <= m <= 12 else f'{y:04d}-01-01'
        return ''
    except (ValueError, TypeError):
        if partial:
            try: return f'{int(year):04d}-01-01'
            except: pass
        return ''


def _find_all_date_tokens(s):
    return [(h.group(1), h.group(2), h.group(3)) for h in _DATE_TOKEN.finditer(s)]


def _resolve_year(yr_str, col):
    yr_int = int(yr_str)
    if yr_int >= 100:
        return yr_int
    century = _century(yr_int, col)
    if century is MANUAL:
        return MANUAL
    return century + yr_int


def _month_from_str(s):
    key = s.lower().rstrip('.')
    return GERMAN_MONTHS.get(key) or SEASON_TO_MONTH.get(key)


def _normalize_single(s, col=None):
    s = s.strip()

    # Unwrap if entire string is parenthesized: "(1885-06-07)", "(1892, 8, 7, )"
    if s.startswith('(') and s.endswith(')') and '(' not in s[1:-1]:
        return _normalize_single(s[1:-1].strip(), col)

    # Preserve year in inline parens before stripping them: "25. Aug. (1920)"
    paren_yr = re.search(r'\((\d{4})\)', s)
    s = re.sub(r'\s*\([^)]*\)', '', s).strip()
    if paren_yr and not re.search(r'\b\d{4}\b', s):
        s = s.rstrip('.').strip() + ' ' + paren_yr.group(1)

    s = re.sub(r'\.$', '', s).strip()
    if not s or s.lower() in _EMPTY_RAW:
        return ''

    s = _ocr_fix(s)

    # Already ISO
    hit = re.match(r'^(\d{4})-(\d{2})-(\d{2})', s)
    if hit:
        return f'{hit.group(1)}-{hit.group(2)}-{hit.group(3)}'

    lower = s.lower()

    # Non-date status tokens
    if any(tok in lower for tok in NON_DATE_TOKENS):
        tokens = _find_all_date_tokens(s)
        if tokens:
            d, mo, yr = tokens[0]
            resolved = _resolve_year(yr, col)
            if resolved is MANUAL: return MANUAL
            return _maybe_flag(_to_iso(d, mo, resolved), resolved)
        month_map = {**GERMAN_MONTHS, **SEASON_TO_MONTH}
        for mon_str, mo in month_map.items():
            if mon_str in lower:
                hit2 = re.search(r'\b(\d{4})\b', s)
                if hit2: return f'{hit2.group(1)}-{mo:02d}-01'
        return ''

    if s == '?':
        return ''
    hit = re.match(r'^\?\.\s*(\d{4})$', s)
    if hit:
        return f'{hit.group(1)}-01-01'

    # Two 4-digit years joined by connector — take the latest
    hit = _ODER_PAT.search(s)
    if hit:
        return f'{max(int(hit.group(1)), int(hit.group(2))):04d}-01-01'

    # "YYYY u. <rest>" — strip leading year+connector, normalize remainder
    hit = re.match(r'^\d{4}\s+u\.\s+(.+)$', s)
    if hit:
        return _normalize_single(hit.group(1), col)

    # "YYYY, <rest>" — leading year with comma; normalize rest with year injected
    hit = re.match(r'^(\d{4}),\s*(.+)$', s)
    if hit:
        yr, rest = hit.group(1), hit.group(2).strip().rstrip('.')
        hit2 = re.match(r'^(\d{1,2})[.\-/,]+(\d{1,2})$', rest)
        if hit2:
            resolved = _resolve_year(yr, col)
            if resolved is MANUAL: return MANUAL
            return _maybe_flag(_to_iso(hit2.group(1), hit2.group(2), resolved, partial=True), resolved)
        candidate = _normalize_single(f'{rest} {yr}', col)
        if candidate:
            return candidate

    # "Jahr" keyword — date context marker; extract year
    if re.search(r'\bjahr\b', lower):
        hit = re.search(r'\b(\d{4})\b', s)
        if hit: return f'{hit.group(1)}-01-01'

    # D.M.YYYY or D.M.YY — accepts . - / , as separators
    hit = re.match(r'^(\d{1,2})[.\-/,]+\s*(\d{1,2})[.\-/,]+\s*(\d{2,4})$', s)
    if hit:
        resolved = _resolve_year(hit.group(3), col)
        if resolved is MANUAL: return MANUAL
        return _maybe_flag(_to_iso(hit.group(1), hit.group(2), resolved, partial=True), resolved)

    # D M. YYYY — space before numeric month, separator after (also handles oversized day)
    hit = re.match(r'^(\d+)\s+(\d{1,2})[.\-]+\s*(\d{2,4})$', s)
    if hit:
        d_raw = int(hit.group(1))
        if d_raw > 31:
            last2 = int(str(d_raw)[-2:])
            day = last2 if 1 <= last2 <= 31 else 1
        else:
            day = d_raw
        resolved = _resolve_year(hit.group(3), col)
        if resolved is MANUAL: return MANUAL
        return _maybe_flag(_to_iso(day, hit.group(2), resolved, partial=True), resolved)

    # Oversized day (3-4 digits).M.YYYY — take last 2 digits of first component
    hit = re.match(r'^(\d{3,4})[.\-/]+(\d{1,2})[.\-/]+(\d{4})$', s)
    if hit:
        last2 = int(hit.group(1)[-2:])
        return _to_iso(last2 if 1 <= last2 <= 31 else 1, hit.group(2), hit.group(3), partial=True)

    # YYYY.M.D
    hit = re.match(r'^(\d{4})[.\-/]+(\d{1,2})[.\-/]+(\d{1,2})$', s)
    if hit:
        return _to_iso(hit.group(3), hit.group(2), hit.group(1), partial=True)

    # MM/DD/YYYY US
    hit = re.match(r'^(\d{1,2})/(\d{1,2})/(\d{4})$', s)
    if hit:
        return _to_iso(hit.group(2), hit.group(1), hit.group(3))

    # D.MYY — merged: "3.275" = day=3, month=2, year=75
    hit = re.match(r'^(\d{1,2})\.(\d)(\d{2})$', s)
    if hit:
        resolved = _resolve_year(hit.group(3), col)
        if resolved is MANUAL: return MANUAL
        return _maybe_flag(_to_iso(hit.group(1), hit.group(2), resolved, partial=True), resolved)

    # D.Roman.YYYY — e.g. "16.XIII.00" → 1900-12-16, "15.IV.33" → 1933-04-15,
    #                     "0.XI.1899" → 1899-11-01, "11.XX.1915" → 1915-12-11
    hit = re.match(r'^(\d{1,2})[\s.\-/,]+([IVXivxLlCc]+)[\s.\-/,]+(\d{2,4})$', s)
    if hit:
        mo_r = _parse_roman(hit.group(2))
        if mo_r:
            resolved = _resolve_year(hit.group(3), col)
            if resolved is MANUAL: return MANUAL
            return _maybe_flag(_to_iso(hit.group(1), min(mo_r, 12), resolved, partial=True), resolved)

    # Roman.Roman.YYYY — e.g. "II.II.1940" → 1940-02-02
    hit = re.match(r'^([IVXivx]+)[.\-/]+([IVXivx]+)[.\-/]+(\d{2,4})$', s)
    if hit:
        day_r = _parse_roman(hit.group(1))
        mo_r  = _parse_roman(hit.group(2))
        if day_r and mo_r:
            resolved = _resolve_year(hit.group(3), col)
            if resolved is MANUAL: return MANUAL
            return _maybe_flag(_to_iso(day_r, mo_r, resolved), resolved)

    # Roman. D. YYYY — Roman=month, numeric=day: "II. 6. 1936" → 1936-02-06
    hit = re.match(r'^([IVXivx]+)[.\-\s]+(\d{1,2})[.\-\s]+(\d{2,4})$', s)
    if hit:
        mo_r = _parse_roman(hit.group(1))
        if mo_r:
            resolved = _resolve_year(hit.group(3), col)
            if resolved is MANUAL: return MANUAL
            return _maybe_flag(_to_iso(hit.group(2), mo_r, resolved), resolved)

    # Roman. Month YYYY — Roman=day, text=month: "II. Sept. 1937" → 1937-09-02
    hit = re.match(rf'^([IVXivx]+)[.\-\s]+([{_UML}]+)\.?[.\-\s,]*(\d{{2,4}})$', s)
    if hit:
        day_r = _parse_roman(hit.group(1))
        mo = _month_from_str(hit.group(2))
        if day_r and mo:
            resolved = _resolve_year(hit.group(3), col)
            if resolved is MANUAL: return MANUAL
            return _maybe_flag(_to_iso(day_r, mo, resolved), resolved)

    # D1/D2 M. YYYY — two alternative days, numeric month: "2/8 3. 1930" → 1930-03-02
    hit = re.match(r'^(\d{1,2})/\d+\s+(\d{1,2})[.\-]+\s*(\d{2,4})$', s)
    if hit:
        mo = int(hit.group(2))
        if 1 <= mo <= 12:
            resolved = _resolve_year(hit.group(3), col)
            if resolved is MANUAL: return MANUAL
            return _maybe_flag(_to_iso(hit.group(1), mo, resolved), resolved)

    # D1/D2. Month YYYY — two alternative days, text month: "14/26. März 1880" → 1880-03-14
    hit = re.match(rf'^(\d{{1,2}})/\d{{1,2}}[.\s]+([{_UML}]+)\.?[.\-\s,]*(\d{{2,4}})$', s)
    if hit:
        mo = _month_from_str(hit.group(2))
        if mo:
            resolved = _resolve_year(hit.group(3), col)
            if resolved is MANUAL: return MANUAL
            return _maybe_flag(_to_iso(hit.group(1), mo, resolved), resolved)

    # D/D YYYY — ambiguous fraction; extract year only: "5/17 1944" → 1944-01-01
    hit = re.match(r'^(\d{1,2})/(\d{1,2})\s+(\d{4})$', s)
    if hit:
        return f'{hit.group(3)}-01-01'

    # Ordinal day + Month + YYYY: "19th March 1913" → 1913-03-19
    hit = re.match(rf'^(\d{{1,2}})(?:st|nd|rd|th)[.,\s]+([{_UML}]+)\.?\s*(\d{{4}})$', s)
    if hit:
        mo = _month_from_str(hit.group(2))
        if mo:
            resolved = _resolve_year(hit.group(3), col)
            if resolved is MANUAL: return MANUAL
            return _maybe_flag(_to_iso(hit.group(1), mo, resolved), resolved)

    # Month oder Month YYYY — take the second (later) month: "Juli oder Aug. 1937"
    hit = re.match(rf'^([{_UML}]+)\.?\s+(?:oder|or)\s+([{_UML}]+)\.?\s*(\d{{4}})$', s)
    if hit:
        mo = _month_from_str(hit.group(2))
        if mo: return f'{hit.group(3)}-{mo:02d}-01'

    # D Month YYYY — includes / in after-month separator for cases like "1.Jul/1883"
    hit = re.match(
        rf'^(\d{{1,2}})[.\-\s,]+([{_UML}]+)\.?[.\-\s,/]*(\d{{2,4}})$', s
    )
    if hit:
        mo = _month_from_str(hit.group(2))
        if mo:
            resolved = _resolve_year(hit.group(3), col)
            if resolved is MANUAL: return MANUAL
            return _maybe_flag(_to_iso(hit.group(1), mo, resolved, partial=True), resolved)

    # Im/In Month YYYY
    hit = re.match(rf'^[Ii][mn]\s+([{_UML}]+)\.?\s*(\d{{4}})$', s)
    if hit:
        mo = _month_from_str(hit.group(1))
        if mo: return f'{hit.group(2)}-{mo:02d}-01'

    # Month D, YYYY — "March 28, 1942", "Sept. 14, 1898", "April 23, 1892"
    hit = re.match(rf'^([{_UML}]+)\.?\s+(\d{{1,2}}),?\s*(\d{{4}})$', s)
    if hit:
        mo = _month_from_str(hit.group(1))
        if mo: return _to_iso(hit.group(2), mo, hit.group(3))

    # Month YYYY or YY — "Feb. 42", "August 1942", "Augl 1901"
    hit = re.match(rf'^([{_UML}]+)\.?\s*(\d{{2,4}})$', s)
    if hit:
        mo = _month_from_str(hit.group(1))
        if mo:
            resolved = _resolve_year(hit.group(2), col)
            if resolved is MANUAL: return MANUAL
            return _maybe_flag(f'{resolved:04d}-{mo:02d}-01', resolved)

    # YYYY D. Month — day before text month: "1889 21. Elul"
    hit = re.match(rf'^(\d{{4}})\s+(\d{{1,2}})[.\s]+([{_UML}]+)\.?$', s)
    if hit:
        mo = _month_from_str(hit.group(3))
        if mo: return _to_iso(hit.group(2), mo, hit.group(1))

    # YYYY Month YYYY — spurious leading year: "1940 März 1939" → 1939-03-01
    hit = re.match(rf'^(\d{{4}})\s+([{_UML}]+)\.?\s*(\d{{4}})$', s)
    if hit:
        mo = _month_from_str(hit.group(2))
        if mo: return f'{hit.group(3)}-{mo:02d}-01'

    # YYYY Month D — "1894 Aug. 18"
    hit = re.match(rf'^(\d{{4}})\s+([{_UML}]+)\.?\s*(\d{{1,2}})$', s)
    if hit:
        mo = _month_from_str(hit.group(2))
        if mo: return _to_iso(hit.group(3), mo, hit.group(1))

    # YYYY. Roman. D — "1897. XII. 23"
    hit = re.match(r'^(\d{4})[.\-\s]+([IVXivx]+)[.\-\s]+(\d{1,2})$', s)
    if hit:
        mo_r = _parse_roman(hit.group(2))
        if mo_r: return _to_iso(hit.group(3), min(mo_r, 12), hit.group(1))

    # YYYY/Month or YYYY Month — "1882/November", "1875 März"
    hit = re.match(rf'^(\d{{4}})[/\s]+([{_UML}]+)\.?$', s)
    if hit:
        mo = _month_from_str(hit.group(2))
        if mo: return f'{hit.group(1)}-{mo:02d}-01'

    # YYYY-Month text — "1903-Sept."
    hit = re.match(rf'^(\d{{4}})[.\-]([{_UML}]+)\.?$', s)
    if hit:
        mo = _month_from_str(hit.group(2))
        if mo: return f'{hit.group(1)}-{mo:02d}-01'

    # YYYY im Month — "1923 im April"
    hit = re.match(rf'^(\d{{4}})\s+[Ii]m\s+([{_UML}]+)\.?$', s)
    if hit:
        mo = _month_from_str(hit.group(2))
        if mo: return f'{hit.group(1)}-{mo:02d}-01'

    # Ende YYYY
    hit = re.match(r'^[Ee]nde\s+(\d{4})$', s)
    if hit: return f'{hit.group(1)}-12-01'

    # M/YYYY or M.YYYY — "11/1906" → 1906-11-01
    hit = re.match(r'^(\d{1,2})[./](\d{4})$', s)
    if hit:
        m = int(hit.group(1))
        if 1 <= m <= 12: return f'{hit.group(2)}-{m:02d}-01'

    # YYYY-MM partial
    hit = re.match(r'^(\d{4})[.\-/](\d{1,2})$', s)
    if hit:
        m = int(hit.group(2))
        if 1 <= m <= 12: return f'{hit.group(1)}-{m:02d}-01'

    # YYYY only
    hit = re.match(r'^(\d{4})$', s)
    if hit: return f'{hit.group(1)}-01-01'

    # M. YY or M. YYYY: "11. 1859", "12.75", "2.1906"
    hit = re.match(r'^(\d{1,2})\.\s*(\d{2}|\d{4})$', s)
    if hit:
        resolved = _resolve_year(hit.group(2), col)
        if resolved is MANUAL: return MANUAL
        return _maybe_flag(f'{resolved:04d}-{int(hit.group(1)):02d}-01', resolved)

    # YYYY-YYYY year range: take first
    hit = re.match(r'^(\d{4})[.\-/]+(\d{4})$', s)
    if hit: return f'{hit.group(1)}-01-01'

    # D.D. Month YYYY — strip leading double-numeric OCR noise: "19.36. Nov. 1936"
    hit = re.match(rf'^\d+[./]+\d+[./\s]+([{_UML}]+)\.?\s*(\d{{2,4}})$', s)
    if hit:
        mo = _month_from_str(hit.group(1))
        if mo:
            resolved = _resolve_year(hit.group(2), col)
            if resolved is MANUAL: return MANUAL
            return _maybe_flag(f'{resolved:04d}-{mo:02d}-01', resolved)

    # Last resort: first D.M.Y token
    tokens = _find_all_date_tokens(s)
    if tokens:
        d, mo, yr = tokens[0]
        resolved = _resolve_year(yr, col)
        if resolved is MANUAL: return MANUAL
        return _maybe_flag(_to_iso(d, mo, resolved, partial=True), resolved)

    # Year-only fallback: extract any plausible 4-digit year from noisy strings
    # e.g. "2. M. 1882", "ca. 1900", "187.1901", "22.1-4/2 1911"
    hit = re.search(r'\b(\d{4})\b', s)
    if hit:
        yr = int(hit.group(1))
        if 1800 <= yr <= 2024:
            return f'{yr:04d}-01-01'

    return ''


_MULTI_LABELED = re.compile(r'[ab1-9]\)\s*')
_TWO_DATES = re.compile(
    r'^(\d{1,2}[.\-/]+\d{1,2}[.\-/]+\d{2,4})\s+(\d{1,2}[.\-/]+\d{1,2}[.\-/]+\d{2,4})$'
)


def normalize_date(raw, col=None):
    """Normalise raw date string to ISO 8601 or MANUAL. Multiple dates -> 'D1 ; D2'."""
    if not raw:
        return ''
    s = raw.strip()
    if not s or s.lower() in _EMPTY_RAW:
        return ''
    s_clean = re.sub(r'\s*\([^)]*\)', '', s).strip()
    if not s_clean:
        s_clean = s.strip('()').strip()

    parts = [p.strip() for p in _MULTI_LABELED.split(s_clean) if p.strip()]
    if len(parts) > 1:
        results = [r for r in [_normalize_single(p, col) for p in parts] if r]
        return ' ; '.join(results) if results else ''

    if ';' in s_clean:
        parts = [p.strip() for p in s_clean.split(';') if p.strip()]
        results = [r for r in [_normalize_single(p, col) for p in parts] if r]
        return ' ; '.join(results) if results else ''

    hit = _TWO_DATES.match(s_clean)
    if hit:
        results = [r for r in [_normalize_single(hit.group(1), col), _normalize_single(hit.group(2), col)] if r]
        return ' ; '.join(results) if results else ''

    return _normalize_single(s, col)


In [5]:
def eval_normalization(split='validation'):
    mask = raw_df['split'] == split
    raw_split  = raw_df[mask].reset_index(drop=True)
    norm_split = norm_df[mask].reset_index(drop=True)

    rows = []
    for i in range(len(raw_split)):
        for col in DATE_COLS:
            raw_val = raw_split.at[i, col]
            gt_val  = norm_split.at[i, col]
            if not raw_val:
                continue
            pred_val = normalize_date(raw_val, col=col)
            rows.append({
                'split': split,
                'record_idx': i,
                'col': col, 'raw': raw_val,
                'pred': pred_val, 'gt': gt_val,
                'correct': pred_val == gt_val,
            })
    df = pd.DataFrame(rows)
    n          = len(df)
    n_correct  = df['correct'].sum()
    n_manual   = (df['pred'] == 'MANUAL').sum()
    has_gt     = df['gt'].notna() & (df['gt'] != '')
    n_gt       = has_gt.sum()
    n_correct_gt = df.loc[has_gt, 'correct'].sum()
    print(
        f"[{split}]  all: {n_correct}/{n} ({100*n_correct/n:.1f}%)"
        f"  |  with GT: {n_correct_gt}/{n_gt} ({100*n_correct_gt/n_gt:.1f}%)"
        f"  |  MANUAL: {n_manual}"
    )
    return df

val_results   = eval_normalization('validation')
train_results = eval_normalization('train')
test_results  = eval_normalization('test')


[validation]  all: 82/91 (90.1%)  |  with GT: 81/84 (96.4%)  |  MANUAL: 0
[train]  all: 443/469 (94.5%)  |  with GT: 430/434 (99.1%)  |  MANUAL: 0
[test]  all: 99/106 (93.4%)  |  with GT: 93/94 (98.9%)  |  MANUAL: 0


In [6]:
errors = test_results[~test_results["correct"]].copy()
print(f"Errors on test: {len(errors)}")
errors[["record_idx", "raw", "pred", "gt", "col"]]


Errors on test: 7


,record_idx,raw,pred,gt,col
8,4,1944,1944-01-01,,VictimDeathDate
24,13,24.6.1941,1941-06-24,,VictimDeathDate
26,15,1.1.76,1876-01-01,1876-01-04,VictimBirthDate
36,22,3.8.1944,1944-08-03,,VictimDeathDate
38,23,25.8.1940,1940-08-25,,VictimDeathDate
70,51,26. März 1903,1903-03-26,,ApplicantBirthDate
105,77,6.3.1941,1941-03-06,,VictimDeathDate


In [7]:
DISPLAY_COLS = [
   'ApplicantBirthDate', 'VictimBirthDate', 'VictimDeathDate'
]


def show_record(i, split='test'):
    """Show all fields of record i (position within split) as a dataframe row."""
    mask     = raw_df['split'] == split
    raw_row  = raw_df[mask].reset_index(drop=True).iloc[i]
    norm_row = norm_df[mask].reset_index(drop=True).iloc[i]

    rows = []
    for col in DISPLAY_COLS:
        raw_val  = raw_row.get(col, '')
        norm_val = norm_row.get(col, '') if col in DATE_COLS else ''
        if col in DATE_COLS:
            pred_val = normalize_date(raw_val, col=col) if raw_val else ''
            rows.append({'field': col, 'raw': raw_val, 'pred': pred_val, 'gt': norm_val})
        else:
            rows.append({'field': col, 'raw': raw_val, 'pred': '', 'gt': '', 'status': ''})

    return pd.DataFrame(rows).set_index('field')

show_record(4)


,raw,pred,gt
field,,,
ApplicantBirthDate,24.4.1858,1858-04-24,1858-04-24
VictimBirthDate,5.6.08,1908-06-05,1908-06-05
VictimDeathDate,1944,1944-01-01,


In [8]:
# Regional folders: BY-BE-Hauptphase, HH-NI-NRW-SH-Hauptphase, RLP-Hauptphase

import json, re
from collections import defaultdict
from pathlib import Path

BZK_DATA_DIR = Path("/home/bzk-data")

REGIONAL_FOLDERS = [
    "BY-BE-Hauptphase",
    "HH-NI-NRW-SH-Hauptphase",
    "RLP-Hauptphase",
]

# raw values that carry no date info and should not count as "unparsed"
_SKIP_RAW = {'nan', 'none', 'null', '', '-', '?'}

# Filenames must follow YYYY_MM_DD_<seq>_0 — others are non-person records
_FNAME_PAT = re.compile(r'^(\d{4})_(\d{2})_(\d{2})_\d+_\d+$')


def filename_to_gt(stem):
    """Parse YYYY_MM_DD_N_0 -> ISO date string (00 month/day treated as 01)."""
    m = _FNAME_PAT.match(stem)
    if not m:
        return None
    yyyy, mm, dd = int(m.group(1)), int(m.group(2)), int(m.group(3))
    if mm == 0:
        return f"{yyyy:04d}-01-01"
    if dd == 0:
        return f"{yyyy:04d}-{mm:02d}-01"
    return f"{yyyy:04d}-{mm:02d}-{dd:02d}"


# Build filename -> (gt_date, region) and group by year
regional_map = {}       
by_year      = defaultdict(dict)  # year_str -> {filename: (gt_date, region)}

for folder in REGIONAL_FOLDERS:
    for img in (BZK_DATA_DIR / folder).glob("*.jpg"):
        gt = filename_to_gt(img.stem)
        if gt is None:
            continue  # skip non-standard filenames
        regional_map[img.name] = (gt, folder)
        year = img.stem.split("_")[0]
        by_year[year][img.name] = (gt, folder)

print(f"Total regional images : {len(regional_map):,}")
print(f"Years spanned         : {min(by_year)} – {max(by_year)}")

# Load matching records (only iterate years present in the regional folders)
rows = []
for year, fname_map in sorted(by_year.items()):
    jsonl = BZK_DATA_DIR / f"{year}.jsonl"
    if not jsonl.exists():
        continue
    with open(jsonl, encoding="utf-8") as fh:
        for line in fh:
            rec   = json.loads(line)
            fname = rec.get("filename", "")
            if fname not in fname_map:
                continue
            gt, region = fname_map[fname]
            raw  = (rec.get("ApplicantBirthDate") or "").strip()
            pred = normalize_date(raw, col="ApplicantBirthDate") if raw and raw.lower() not in _SKIP_RAW else ""
            rows.append({
                "region":   region,
                "filename": fname,
                "gt":       gt,
                "raw":      raw,
                "pred":     pred,
                "correct":  pred == gt,
                "manual":   pred == "MANUAL",
                "empty":    pred == "" and bool(raw) and raw.lower() not in _SKIP_RAW,
            })

regional_df = pd.DataFrame(rows)
print(f"Records matched       : {len(regional_df):,}")
print(f"  with raw date value : {(regional_df['raw'] != '').sum():,}")
print(f"  no raw value        : {(regional_df['raw'] == '').sum():,}")
regional_df.head(5)

Total regional images : 696,878
Years spanned         : 1819 – 1994
Records matched       : 696,935
  with raw date value : 694,099
  no raw value        : 2,836


,region,filename,gt,raw,pred,correct,manual,empty
0,BY-BE-Hauptphase,1819_06_06_1_0.jpg,1819-06-06,6.6.1819,1819-06-06,True,False,False
1,BY-BE-Hauptphase,1842_09_27_1_0.jpg,1842-09-27,,,False,False,False
2,BY-BE-Hauptphase,1853_04_20_2_0.jpg,1853-04-20,20.4.1853,1853-04-20,True,False,False
3,RLP-Hauptphase,1854_12_30_1_0.jpg,1854-12-30,30.12.1894,1894-12-30,False,False,False
4,RLP-Hauptphase,1855_00_00_1_0.jpg,1855-01-01,1855,1855-01-01,True,False,False


In [9]:
# Evaluate normalization on regional data 

def eval_regional(df, label="ALL"):
    sub = df[df["raw"] != ""].copy()
    n         = len(sub)
    if n == 0:
        print(f"[{label}]  no records with raw date value")
        return
    n_correct = sub["correct"].sum()
    n_manual  = sub["manual"].sum()
    n_empty   = sub["empty"].sum()
    print(
        f"[{label:<30}]  n={n:>7,}  "
        f"correct={n_correct:>7,} ({100*n_correct/n:5.1f}%)  "
        f"MANUAL={n_manual:>5,}  unparsed={n_empty:>5,}"
    )

eval_regional(regional_df, "ALL regions")
print()
for folder in REGIONAL_FOLDERS:
    sub = regional_df[regional_df["region"] == folder]
    eval_regional(sub, folder)


[ALL regions                   ]  n=694,099  correct=642,004 ( 92.5%)  MANUAL=    0  unparsed=   86

[BY-BE-Hauptphase              ]  n=221,558  correct=215,015 ( 97.0%)  MANUAL=    0  unparsed=    5
[HH-NI-NRW-SH-Hauptphase       ]  n=254,008  correct=221,595 ( 87.2%)  MANUAL=    0  unparsed=   64
[RLP-Hauptphase                ]  n=218,533  correct=205,394 ( 94.0%)  MANUAL=    0  unparsed=   17


In [10]:
# Browse errors and edge-cases on regional data 
reg_with_raw = regional_df[regional_df["raw"] != ""].copy()

errors_reg  = reg_with_raw[~reg_with_raw["correct"]]
manual_reg  = reg_with_raw[reg_with_raw["manual"]]
empty_reg   = reg_with_raw[reg_with_raw["empty"]]

print(f"Wrong predictions : {len(errors_reg):,}")
print(f"MANUAL review     : {len(manual_reg):,}")
print(f"Could not parse   : {len(empty_reg):,}")
print()
display(errors_reg[["region","filename","raw","pred","gt"]])


Wrong predictions : 52,095
MANUAL review     : 0
Could not parse   : 86



,region,filename,raw,pred,gt
3,RLP-Hauptphase,1854_12_30_1_0.jpg,30.12.1894,1894-12-30,1854-12-30
7,BY-BE-Hauptphase,1857_12_28_3_0.jpg,28.11.57,1857-11-28,1857-12-28
11,BY-BE-Hauptphase,1857_12_28_2_0.jpg,28.12.1897,1897-12-28,1857-12-28
20,HH-NI-NRW-SH-Hauptphase,1859_11_26_1_0.jpg,21.11.1909,1909-11-21,1859-11-26
23,HH-NI-NRW-SH-Hauptphase,1859_11_26_2_0.jpg,11. 1859,1859-11-01,1859-11-26
...,...,...,...,...,...
696918,HH-NI-NRW-SH-Hauptphase,1966_05_29_1_0.jpg,29.5.66,1866-05-29,1966-05-29
696920,BY-BE-Hauptphase,1967_08_24_1_0.jpg,24.8.1907,1907-08-24,1967-08-24
696921,BY-BE-Hauptphase,1968_07_02_1_0.jpg,2.7.68,1868-07-02,1968-07-02
696923,BY-BE-Hauptphase,1968_06_22_1_0.jpg,22.6.18,1918-06-22,1968-06-22


In [11]:
empty_reg[["region","filename","raw","pred","gt"]]

,region,filename,raw,pred,gt
11747,HH-NI-NRW-SH-Hauptphase,1877_06_04_18_0.jpg,4.6.19XX,,1877-06-04
12598,HH-NI-NRW-SH-Hauptphase,1878_11_08_2_0.jpg,8. M. 18,,1878-11-08
23706,HH-NI-NRW-SH-Hauptphase,1881_08_14_3_0.jpg,14 de agosto 1.881,,1881-08-14
26408,HH-NI-NRW-SH-Hauptphase,1882_01_01_3_0.jpg,11.8.2,,1882-01-01
29803,HH-NI-NRW-SH-Hauptphase,1882_09_28_8_0.jpg,20.5.8,,1882-09-28
...,...,...,...,...,...
626816,HH-NI-NRW-SH-Hauptphase,1930_01_07_8_0.jpg,7.13.0,,1930-01-07
643963,RLP-Hauptphase,1933_08_03_23_0.jpg,3(2)1933,,1933-08-03
660641,HH-NI-NRW-SH-Hauptphase,1935_04_02_37_0.jpg,24.3.5,,1935-04-02
661593,HH-NI-NRW-SH-Hauptphase,1935_07_11_18_0.jpg,117.35,,1935-07-11


In [12]:
# ── Sanity-check: all targeted cases ──────────────────────────────────────────
cases = [
    # batch 1
    ("11. 1859",              "ApplicantBirthDate", "1859-11-01"),
    ("5.0.66",                "ApplicantBirthDate", "1866-01-01"),
    ("75.70.1953",            "ApplicantBirthDate", "1953-07-01"),
    ("1886-1865",             "ApplicantBirthDate", "1886-01-01"),
    ("nan",                   "ApplicantBirthDate", ""),
    ("2-II-1870",             "ApplicantBirthDate", "1870-02-02"),
    ("1875 März",             "ApplicantBirthDate", "1875-03-01"),
    ("1974.3.1",              "ApplicantBirthDate", "1974-03-01"),
    ("1874.8.74",             "ApplicantBirthDate", "1874-08-01"),
    ("Im Juli 1942",          "VictimDeathDate",    "1942-07-01"),
    ("1944-11",               "VictimDeathDate",    "1944-11-01"),
    ("22-Oktober-1946",       "VictimBirthDate",    "1946-10-22"),
    ("1943 oder 1944",        "VictimDeathDate",    "1944-01-01"),
    # batch 2
    ("1.0kt.1871",            "ApplicantBirthDate", "1871-10-01"),
    ("1894 Aug. 18",          "ApplicantBirthDate", "1894-08-18"),
    ("12.75",                 "ApplicantBirthDate", "1875-12-01"),
    ("3.275",                 "ApplicantBirthDate", "1875-02-03"),
    ("1. J. 1877",            "ApplicantBirthDate", "1877-01-01"),
    ("11. ug. 1942",          "VictimDeathDate",    "1942-08-11"),
    ("18 IX, 1942",           "VictimDeathDate",    "1942-09-18"),
    ("Feb. 42",               "VictimDeathDate",    "1942-02-01"),
    ("March 28, 1942",        "VictimDeathDate",    "1942-03-28"),
    # batch 3
    ("Jänner 1877",           "ApplicantBirthDate", "1877-01-01"),
    ("1914.10.1877",          "ApplicantBirthDate", "1877-10-14"),
    ("1940 März 1939",        "VictimDeathDate",    "1939-03-01"),
    ("1, III. 1940",          "VictimDeathDate",    "1940-03-01"),
    ("II.II.1940",            "VictimDeathDate",    "1940-02-02"),
    # batch 4
    ("116.7.1936",            "ApplicantBirthDate", "1936-07-16"),
    ("1913.7.1936",           "ApplicantBirthDate", "1936-07-13"),
    ("1934 u. 1. Aug. 1938",  "VictimDeathDate",    "1938-08-01"),
    ("II. Sept. 1937",        "VictimDeathDate",    "1937-09-02"),
    ("14/26. März 1880",      "ApplicantBirthDate", "1880-03-14"),
    # batch 5
    ("1882/November",         "ApplicantBirthDate", "1882-11-01"),
    ("II. 6. 1936",           "VictimDeathDate",    "1936-02-06"),
    ("19.36. Nov. 1936",      "VictimDeathDate",    "1936-11-01"),
    # batch 6
    ("2. Novl 1883",          "ApplicantBirthDate", "1883-11-02"),
    ("7,6,1885",              "ApplicantBirthDate", "1885-06-07"),
    ("(1885-06-07)",          "ApplicantBirthDate", "1885-06-07"),
    ("1888, 15. März",        "ApplicantBirthDate", "1888-03-15"),
    ("1889 21. Elul",         "ApplicantBirthDate", "1889-06-21"),
    ("1897. XII. 23",         "ApplicantBirthDate", "1897-12-23"),
    ("1890, 31.1.",            "ApplicantBirthDate", "1890-01-31"),
    ("1893, Sept. 14",        "ApplicantBirthDate", "1893-09-14"),
    ("8, 7, 1898",            "ApplicantBirthDate", "1898-07-08"),
    ("16.XIII.00",            "ApplicantBirthDate", "1900-12-16"),
    ("3,10. 1898",            "ApplicantBirthDate", "1898-10-03"),
    ("24. K. 1900",           "ApplicantBirthDate", "1900-10-24"),
    ("1903, 7. Oktober",      "ApplicantBirthDate", "1903-10-07"),
    ("8 10. 1905",            "ApplicantBirthDate", "1905-10-08"),
    ("22,2,1906",             "ApplicantBirthDate", "1906-02-22"),
    ("12 Dezemb 1915",        "ApplicantBirthDate", "1915-12-12"),
    ("25. Jani 1916",         "ApplicantBirthDate", "1916-06-25"),
    ("81 4. 29",              "VictimDeathDate",    "1929-04-01"),
    ("24 3-1935",             "ApplicantBirthDate", "1935-03-24"),
    ("1. Jahr 1891",          "ApplicantBirthDate", "1891-01-01"),
    ("5/17 1944",             "VictimDeathDate",    "1944-01-01"),
    ("Augl 1901",             "ApplicantBirthDate", "1901-08-01"),
    ("1903-Sept.",            "ApplicantBirthDate", "1903-09-01"),
    ("1920, April",           "ApplicantBirthDate", "1920-04-01"),
    ("25. Aug. (1920)",        "ApplicantBirthDate", "1920-08-25"),
    ("1923 im April",         "ApplicantBirthDate", "1923-04-01"),
    ("2/8 3. 1930",           "ApplicantBirthDate", "1930-03-02"),
    ("15.1V.33",              "VictimDeathDate",    "1933-04-15"),
    ("74. März 1897",         "ApplicantBirthDate", "1897-03-01"),
    ("0. Januar 1914",        "ApplicantBirthDate", "1914-01-01"),
    ("April 23, 1892",        "ApplicantBirthDate", "1892-04-23"),
    ("Sept. 14, 1898",        "ApplicantBirthDate", "1898-09-14"),
    # batch 7
    ("1.8ul/1883",            "ApplicantBirthDate", "1883-07-01"),
    ("2. M. 1882",            "ApplicantBirthDate", "1882-01-01"),
    ("(1892, 8, 7, )",        "ApplicantBirthDate", "1892-07-08"),
    ("ca. 1900",              "ApplicantBirthDate", "1900-01-01"),
    ("187.1901",              "ApplicantBirthDate", "1901-01-01"),
    ("22.1-4/2 1911",         "ApplicantBirthDate", "1911-01-01"),
    ("11.XX.1915",            "ApplicantBirthDate", "1915-12-11"),
    ("25.Jan. oder Feb.1919", "ApplicantBirthDate", "1919-01-01"),
    ("Juli oder Aug. 1937",   "VictimDeathDate",    "1937-08-01"),
    ("0.XI.1899",             "ApplicantBirthDate", "1899-11-01"),
    ("11/1906",               "ApplicantBirthDate", "1906-11-01"),
    ("19th March 1913",       "ApplicantBirthDate", "1913-03-19"),
]
print(f"{'raw':<30} {'col':<22} {'pred':<15} {'expected':<15} ok")
print("-" * 90)
all_ok = True
for raw, col, expected in cases:
    pred = normalize_date(raw, col=col)
    ok   = pred == expected
    all_ok = all_ok and ok
    print(f"{raw:<30} {col:<22} {pred:<15} {expected:<15} {'✓' if ok else '✗  ← FAIL'}")
print()
print(f"Total: {len(cases)} cases — {'All passing ✓' if all_ok else 'Some FAILING — check above'}")

raw                            col                    pred            expected        ok
------------------------------------------------------------------------------------------
11. 1859                       ApplicantBirthDate     1859-11-01      1859-11-01      ✓
5.0.66                         ApplicantBirthDate     1866-01-01      1866-01-01      ✓
75.70.1953                     ApplicantBirthDate     1953-07-01      1953-07-01      ✓
1886-1865                      ApplicantBirthDate     1886-01-01      1886-01-01      ✓
nan                            ApplicantBirthDate                                     ✓
2-II-1870                      ApplicantBirthDate     1870-02-02      1870-02-02      ✓
1875 März                      ApplicantBirthDate     1875-03-01      1875-03-01      ✓
1974.3.1                       ApplicantBirthDate     1974-03-01      1974-03-01      ✓
1874.8.74                      ApplicantBirthDate     1874-08-01      1874-08-01      ✓
Im Juli 1942                